# Doublet Consensus — Cross-Method Comparison & Removal

Loads the shared object with doublet annotations from all integration methods,
compares overlap, removes consensus doublets, and saves a clean object.

**Steps:**
1. Load shared object
2. Count doublets per method
3. Find overlap across all methods
4. Remove consensus doublets
5. Save clean object

In [ ]:
SHARED_PATH = "/vol/disk/ubuntu/master_practicum_cytokines/data/data_for_practicum_preprocessed_doublet_clusters.h5ad"
OUT_PATH    = "/vol/disk/ubuntu/master_practicum_cytokines/data/data_for_practicum_consensus_doublets_removed.h5ad"

import os
print(f"Input:  {SHARED_PATH}")
print(f"Output: {OUT_PATH}")

In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt
import pandas as pd
import scanpy as sc
from upsetplot import UpSet, from_memberships

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=100, facecolor="white", frameon=False)

## Step 1 — Load shared object

In [ ]:
adata = sc.read_h5ad(SHARED_PATH)
print(f"Cells: {adata.n_obs:,}   Genes: {adata.n_vars:,}")

doublet_cols = [c for c in adata.obs.columns if c.endswith("_doublet")]
print(f"\nDoublet columns found: {doublet_cols}")

## Step 2 — Doublets detected per method

In [ ]:
counts = {col: adata.obs[col].sum() for col in doublet_cols}

print("Doublets detected per method:")
for col, n in counts.items():
    print(f"  {col}: {n:,} ({n/adata.n_obs*100:.1f}%)")

# Bar plot
fig, ax = plt.subplots(figsize=(max(5, len(doublet_cols) * 1.2), 4))
ax.bar(counts.keys(), counts.values(), color="steelblue")
ax.set_ylabel("Number of doublet cells")
ax.set_title("Doublets detected per integration method")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

## Step 3 — Overlap across methods

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

method_names = [col.replace("_doublet", "") for col in doublet_cols]

doublet_df = adata.obs[doublet_cols].copy()
doublet_df.columns = method_names

from upsetplot import UpSet, from_indicators
data = from_indicators(method_names, data=doublet_df[doublet_df.any(axis=1)])

upset = UpSet(data, subset_size="count", show_counts=False, sort_by="cardinality")
upset.plot()
plt.savefig("/vol/disk/ubuntu/master_practicum_cytokines/lisa/doublet_upset.png", dpi=100)
plt.show()

# Consensus — flagged by ALL methods
consensus = adata.obs[doublet_cols].all(axis=1)
print(f"Cells flagged by ALL methods: {consensus.sum():,} ({consensus.sum()/adata.n_obs*100:.1f}%)")

## Step 4 — Remove consensus doublets & save

In [ ]:
adata_clean = adata[~consensus].copy()

print(f"Cells before removal: {adata.n_obs:,}")
print(f"Cells removed:        {consensus.sum():,}")
print(f"Cells after removal:  {adata_clean.n_obs:,}")

for col in adata_clean.obs.columns:
    if adata_clean.obs[col].dtype == object:
        adata_clean.obs[col] = adata_clean.obs[col].astype(str)

adata_clean.write_h5ad(OUT_PATH)
print(f"\nSaved to: {OUT_PATH}")